# OMM 데이터 분석 — 3팀 (⑤ 트렌드 키워드 · ⑥ 영양 효율)

**사용법**
1. 맨 위 `[0단계] 준비` 셀부터 차례로 실행 (Shift+Enter)
2. 분석 코드를 고치려면 `%%writefile` 셀 안에서 고친 뒤 그 셀을 실행 → `report/analyses/` 의 내 파일이 저장돼요
3. 바로 아래 `[실행]` 셀을 실행하면 고친 코드로 그림이 다시 그려지고, `[데이터 확인]` 셀에서 불러온 DataFrame 을 볼 수 있어요
4. 사이트 반영: 관리자 화면 › 보고서 차트 › **다시 그리기** (또는 매일 18:30 자동). 데이터 분석 탭의 코드 칸은 저장하는 즉시 바뀌어요

> 다른 팀 파일(`analyses/` 의 다른 번호)과 `common.py` 는 고치지 마세요. 공통 부분을 바꿔야 하면 팀에 먼저 알려 주세요.

In [ ]:
# [0단계] 준비 — 공통 코드(common.py) 불러오기 · DB 접속 정보
import os, sys, importlib
REPORT = os.getenv('OMM_REPORT', '/opt/pipeline/api/report')   # back1 기준 분석 코드 폴더
sys.path.insert(0, REPORT)

# DB 접속: API 와 같은 읽기 전용 계정. 비밀번호는 화면에 출력하지 않아요
if not os.getenv('PGPASSWORD'):
    try:   # 설치 때 만든 읽기 전용 DB 접속 파일 (팀원 계정만 읽을 수 있음)
        for line in open(os.path.join(REPORT, '.pgenv'), encoding='utf-8'):
            k, _, v = line.strip().partition('=')
            if k.startswith('PG') and v: os.environ.setdefault(k, v.strip('"\''))
    except (FileNotFoundError, PermissionError):
        from getpass import getpass
        os.environ['PGPASSWORD'] = getpass('DB 비밀번호 (omm_api): ')

# 노트북에서 그린 그림은 notebooks/_out 에 저장 (사이트 그림은 건드리지 않음)
os.environ['REPORT_DIR'] = os.path.join(REPORT, 'notebooks', '_out')
os.makedirs(os.environ['REPORT_DIR'], exist_ok=True)

import common
common.SHOW = True          # save() 할 때 그림을 셀 아래에 바로 보여 주기
from common import *
print('준비 완료')

## ⑤ 트렌드 키워드
최근 레시피 트렌드 키워드 분석 (예: 로제 · 두바이) — YouTube Data API

- 파일: `report/analyses/a5_trend.py` · 함수: `chart5()`
- 그림 파일: `notebooks/_out/` 에 PNG · SVG 로 저장

In [ ]:
%%writefile ../analyses/a5_trend.py
"""⑤ 트렌드 키워드 — YouTube Data API 로 본 최근 레시피 트렌드 키워드 (예: 로제 · 두바이)
담당: 3팀 (⑤⑥) · 이 파일만 고치면 돼요.
  - 노트북 notebooks/3팀_⑤⑥.ipynb 에서 고치고 바로 실행해 볼 수 있어요.
  - 서버는 매일 18:30 charts_v4.py 가 이 파일의 chart5() 을 불러 그림을 다시 그려요.
  - 관리자 화면 › 데이터 분석 탭의 '차트 그리기' 코드 칸에 이 파일 내용이 그대로 보여요.
"""
from common import *

# [1단계] DB에서 데이터 불러오기 (q = SQL → pandas DataFrame)
# [2단계] pandas 로 가공
# [3단계] matplotlib 으로 그리고 save() 로 저장 (PNG + SVG)

def chart5():
    d = q("""SELECT k.keyword, kt.ym, kt.yt_video_cnt AS yt, kt.yt_view_sum AS views, kt.recipe_cnt AS rc
             FROM omm_keyword_trend_monthly kt JOIN omm_trend_keywords k USING (keyword_id) WHERE kt.ym <= date_trunc('month', current_date) ORDER BY kt.ym""")
    if d.empty: return empty('5_트렌드.png', 5, '최근 레시피 트렌드 키워드', 'YouTube 수집 후 다시 실행')
    d['ym'] = pd.to_datetime(d.ym)
    top = d.groupby('keyword').views.sum().sort_values(ascending=False).head(4).index.tolist()
    color = {k: S[i] for i, k in enumerate(top)}
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
    for k in top:
        g = d[d.keyword == k]
        a1.plot(g.ym, g.views / 1e4, color=color[k], lw=2, marker='o', ms=4, label=k)
        a1.annotate(k, (g.ym.iloc[-1], g.views.iloc[-1] / 1e4), xytext=(6, 0), textcoords='offset points', color=INK, va='center')
        a2.plot(g.ym, g.rc, color=color[k], lw=2, marker='o', ms=4, label=k)
    a1.set_title('⑤ 트렌드 키워드 — YouTube 월 게시 영상 조회수 합 (만 회)'); a1.legend(loc='upper left', ncol=4)
    a2.set_title('같은 키워드가 제목에 든 만개 신규 레시피 수'); a2.set_ylabel('레시피 수')
    note(a2, '조회수 합 상위 4개 키워드 · 키워드 × 월마다 조회수 상위 50개 영상 기준 · 두 지표는 단위가 달라 따로 그림')
    g = d.groupby('keyword').views.sum().sort_values().tail(12) / 1e4
    fig2, ax = plt.subplots(figsize=(9, 0.5 + len(g) * 0.45))
    ax.barh(g.index, g.values, color=[color.get(k, MUTED) for k in g.index], height=0.6)
    for y, v in enumerate(g.values): ax.text(v, y, f'  {v:,.0f}만', va='center', color=INK2, fontsize=11)
    ax.set_title('키워드별 전체 조회수 합 (만 회)'); ax.grid(axis='y', visible=False)
    save(fig, '5_트렌드.png', 5, '트렌드 키워드 월별 흐름', 'YouTube 조회수와 만개 레시피 수를 같은 색으로 비교')
    save(fig2, '5_트렌드_순위.png', 5, '트렌드 키워드 순위', '전체 기간 조회수 합')

if __name__ == '__main__':
    chart5()

In [ ]:
# [실행] 저장한 파일을 다시 읽어서 그림 그리기
from analyses import a5_trend
importlib.reload(a5_trend)
common.queries()          # 이전 기록 비우기
a5_trend.chart5()

In [ ]:
# [데이터 확인] ⑤ 트렌드 키워드 이 DB 에서 불러온 DataFrame (SQL → pandas)
for i, (sql, df) in enumerate(common.queries(), 1):
    print(f'── 불러온 데이터 {i} · {len(df):,}행 × {df.shape[1]}열')
    display(df.head(10))

## ⑥ 영양 효율
재료비 대비 영양 효율성 — 공공데이터(KAMIS 소매 · 도매, 참가격, 식약처 영양성분)

- 파일: `report/analyses/a6_value.py` · 함수: `chart6()`
- 그림 파일: `notebooks/_out/` 에 PNG · SVG 로 저장

In [ ]:
%%writefile ../analyses/a6_value.py
"""⑥ 영양 효율 — 재료비 대비 영양 효율 (공공데이터: KAMIS · 참가격 · 식약처)
담당: 3팀 (⑤⑥) · 이 파일만 고치면 돼요.
  - 노트북 notebooks/3팀_⑤⑥.ipynb 에서 고치고 바로 실행해 볼 수 있어요.
  - 서버는 매일 18:30 charts_v4.py 가 이 파일의 chart6() 을 불러 그림을 다시 그려요.
  - 관리자 화면 › 데이터 분석 탭의 '차트 그리기' 코드 칸에 이 파일 내용이 그대로 보여요.
"""
from common import *

# [1단계] DB에서 데이터 불러오기 (q = SQL → pandas DataFrame)
# [2단계] pandas 로 가공
# [3단계] matplotlib 으로 그리고 save() 로 저장 (PNG + SVG)

def chart6():
    d = q("""SELECT i.name, nf.protein_g / pm.price_per_100g * 1000 AS p, nf.energy_kcal / pm.price_per_100g * 1000 AS k
             FROM omm_ingredients i JOIN omm_nutrition_foods nf ON nf.food_cd = i.nutr_food_cd
             JOIN omm_price_monthly pm ON pm.ingredient_id = i.ingredient_id AND pm.market_type = 'MARKET_RETAIL' AND pm.ym = (SELECT max(ym) FROM omm_price_monthly)
             WHERE pm.price_per_100g > 0 AND nf.protein_g IS NOT NULL""")
    unit = '재료'
    if d.empty:                                   # 재료별 영양성분이 없으면(팀 공유 데이터) 메뉴 영양성분 × 레시피 재료비로
        d = q("""SELECT m.name, percentile_cont(0.5) WITHIN GROUP (ORDER BY rm.protein_per_1000won) AS p,
                        percentile_cont(0.5) WITHIN GROUP (ORDER BY rm.kcal_per_1000won) AS k
                 FROM omm_recipe_metrics rm JOIN omm_recipes r USING (recipe_id) JOIN omm_menus m USING (menu_id)
                 WHERE rm.protein_per_1000won IS NOT NULL AND rm.priced_ratio >= 0.6 AND rm.cost_per_serving >= 500
                 GROUP BY m.name HAVING count(*) >= 5""")
        unit = '메뉴'
    if d.empty: return empty('6_영양효율.png', 6, '재료비 대비 영양 효율', '영양성분과 재료비가 함께 있는 재료 · 메뉴가 없음')
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, max(6, min(len(d), 20) * 0.35)))
    t = d.sort_values('p').tail(20)
    ax.barh(t.name, t.p, color=S[0], height=0.6)
    for y, v in enumerate(t.p): ax.text(v, y, f'  {v:.1f}g', va='center', fontsize=11, color=INK2)
    ax.set_title(f'⑥ 1,000원으로 살 수 있는 단백질 (g) · {unit}별'); ax.grid(axis='y', visible=False)
    ax2.scatter(d.k, d.p, s=50, color=S[0], alpha=0.7, edgecolor=SURF, linewidth=1)
    mk, mp = d.k.median(), d.p.median()
    ax2.axvline(mk, color=MUTED, ls=':', lw=1); ax2.axhline(mp, color=MUTED, ls=':', lw=1)
    lab = pd.concat([d.nlargest(6, 'p'), d.nlargest(4, 'k'), d.nsmallest(3, 'p')]).drop_duplicates('name')
    for _, r in lab.iterrows(): ax2.annotate(r['name'], (r.k, r.p), xytext=(5, 4), textcoords='offset points', fontsize=11, color=INK)
    ax2.text(0.98, 0.98, '단백질 · 열량 모두 가성비 ↑', transform=ax2.transAxes, ha='right', va='top', color=INK2, fontsize=11)
    ax2.text(0.02, 0.02, '둘 다 비쌈', transform=ax2.transAxes, color=INK2, fontsize=11)
    ax2.set_xlabel('1,000원당 열량 (kcal)'); ax2.set_ylabel('1,000원당 단백질 (g)'); ax2.set_title('열량 · 단백질 가성비 사분면')
    note(ax, '최근 월 KAMIS 소매가 · 식약처 100g당 영양성분 · 가격과 영양이 모두 연결된 재료만' if unit == '재료' else
             '메뉴별 식약처 1인분 영양성분 ÷ 레시피 1인분 재료비 · 레시피 5개 이상 메뉴의 중앙값 (가격 붙은 재료 60% 이상 · 1인분 500원 이상)')
    save(fig, '6_영양효율.png', 6, '재료비 대비 영양 효율', '1,000원당 단백질 · 열량')

if __name__ == '__main__':
    chart6()

In [ ]:
# [실행] 저장한 파일을 다시 읽어서 그림 그리기
from analyses import a6_value
importlib.reload(a6_value)
common.queries()          # 이전 기록 비우기
a6_value.chart6()

In [ ]:
# [데이터 확인] ⑥ 영양 효율 이 DB 에서 불러온 DataFrame (SQL → pandas)
for i, (sql, df) in enumerate(common.queries(), 1):
    print(f'── 불러온 데이터 {i} · {len(df):,}행 × {df.shape[1]}열')
    display(df.head(10))

---
### 결과 정리 (직접 작성)
- 분석 결과:
- 서비스에 활용한 곳:
- 수익 모델과의 연결: